**Оценка модели регрессии. Прогноз января 2026 г.**

In [7]:
# %cd D:\Python_2\MLogic_project\Yuperio\prognoz_prodaj_12_2025
%cd /Volumes/external_disk/MLogic/MLogic_project/Yuperio/prognoz_prodaj_12_2025

/Volumes/external_disk/MLogic/MLogic_project/Yuperio/prognoz_prodaj_12_2025


In [8]:
import pandas as pd
import numpy as np

from utils import evaluate_model

In [9]:
df_sale_true = pd.read_csv('etl/sale_01_2026.csv')
df_sale_true = df_sale_true[(df_sale_true['exit_type']=='Продажа') & (df_sale_true['Territory'].str.contains('moscow', case=False, na=False)) ]
df_sale_true

,location_mdlp_id,Адрес ЛПУ,Territory,exit_type,2026-01-01
0,166,"Москва г, Митинский 3-й пер., 4",MR PC BU CVM Moscow_3,Продажа,24.0
1,167,"Серпухов г, Московское ш., 64",MR PC BU CVM Moscow_11,Продажа,8.0
2,169,"Москва г, Маршала Федоренко ул., 6",MR PC BU CVM Moscow_SAO,Продажа,19.0
3,170,"Москва г, Скульптора Мухиной ул., 14",MR PC BU CVM Moscow_2,Продажа,14.0
4,171,"Москва г, Героев Панфиловцев ул., 8",MR PC BU CVM Moscow_3,Продажа,3.0
...,...,...,...,...,...
50352,535646,"Юность п, нет, 2",MR PC BU CVM Moscow_UAO_2,Продажа,5.0
50382,535785,"Москва г, Вольская 1-я ул., 3",MR PC BU CVM Moscow_UVAO_1,Продажа,NaN
50383,535786,"Одинцово г, Красногорское ш., 15",MR PC BU CVM Moscow_SZAO_1,Продажа,4.0
50394,535826,"Москва г, Новаторов ул., 5",MR PC BU CVM Moscow_5,Продажа,2.0


In [10]:
df_sale_predict = pd.read_csv('result_predict.csv')
df_sale_predict

,mdlp_id,sale
0,166,25
1,167,4
2,168,0
3,169,25
4,170,15
...,...,...
6888,535646,3
6889,535785,2
6890,535786,2
6891,535826,4


In [11]:
df_merge = pd.merge(
    df_sale_true,
    df_sale_predict,
    left_on='location_mdlp_id',
    right_on='mdlp_id',
    how='inner'  # inner join (только совпадающие записи)
)

de_merge = df_merge[['mdlp_id', '2026-01-01', 'sale']]
de_merge = de_merge.rename(columns={'2026-01-01': 'fact', 'sale': 'predict'})
de_merge = de_merge.fillna(0)
de_merge

,mdlp_id,fact,predict
0,166,24.0,25
1,167,8.0,4
2,169,19.0,25
3,170,14.0,15
4,171,3.0,2
...,...,...,...
5276,535646,5.0,3
5277,535785,0.0,2
5278,535786,4.0,2
5279,535826,2.0,4


In [12]:
fact = de_merge['fact'].to_list()
predict = de_merge['predict'].to_list()

evaluate_model(fact, predict, 'LightGBM')


LightGBM Результаты:
Среднеквадратичная ошибка (MSE): 35.2248
Корень из MSE (RMSE): 5.9350
Средняя абсолютная ошибка (MAE): 3.1024
Симетричная средняя абсолютная процентная ошибка (SMAPE): 63.90%
Коэффициент детерминации (R²): 0.6686
